In [15]:
import sys
sys.path.append('..')

import importlib
import agents.critic as cr
importlib.reload(cr)
from agents.critic import run_critic, route_from_critic

print("critic imported")

critic imported


In [16]:
import importlib
import agents.critic as cr
import agents.narrator as na
importlib.reload(cr)
importlib.reload(na)
from agents.critic import run_critic, route_from_critic
from agents.narrator import run_narrator
print("reloaded")

reloaded


In [17]:
# test the LLM call with a manually built state
# this way you verify the prompt works before running the full graph

fake_state = {
    "prediction": "mel",
    "confidence": 0.62,
    "loop_count": 0,
    "shap_result": {
        "status": "success",
        "mean_shap": 0.000089,
        "max_shap": 0.003421,
        "interpretation": "Low feature attribution — decision distributed broadly across image"
    },
    "gradcam_result": {
        "status": "success",
        "attention_region": "center of the lesion",
        "max_attention": 0.94,
        "interpretation": "The model attention was tightly focused on a small region with high gradient signal strength"
    }
}

print("testing Critic LLM call with fake data...")
print("this should find a contradiction: SHAP is low but Grad-CAM is high\n")

output = run_critic(fake_state)
print("\ncritique:", output['critique'])
print("contradictions:", output['contradictions'])
print("loop_count:", output['loop_count'])

testing Critic LLM call with fake data...
this should find a contradiction: SHAP is low but Grad-CAM is high

  [Critic node] reviewing SHAP and Grad-CAM with LLM...
  [Critic node] done. contradictions=1, loop=0
    - The low feature attribution from SHAP contradicts the high attention from Grad-CAM, suggesting that the model's decision is not as distributed as SHAP suggests.

critique: Agreement: Both SHAP and Grad-CAM indicate that the model's prediction is based on the image, with SHAP showing low feature attribution and Grad-CAM focusing on a specific region. Confidence assessment: The model's confidence of 62% seems moderate but does not strongly match the evidence, as the high attention from Grad-CAM would typically suggest higher confidence. Next focus: Investigate the specific region highlighted by Grad-CAM to understand why the model's attention was so focused and how it relates to the predicted diagnosis of melanoma.
contradictions: ["The low feature attribution from SHAP co

In [18]:
updated_state = {**fake_state, **output}

route = route_from_critic(updated_state)
print("router went to:", route)
print("expected: planner (because contradiction found and loop_count < 2)")

  [Critic router] contradiction found, looping back to planner
router went to: planner
expected: planner (because contradiction found and loop_count < 2)


In [19]:
clean_state = {
    "prediction": "nv",
    "confidence": 0.99,
    "loop_count": 0,
    "shap_result": {
        "status": "success",
        "mean_shap": 0.0045,
        "max_shap": 0.021,
        "interpretation": "High feature concentration — model focused on specific localised region"
    },
    "gradcam_result": {
        "status": "success",
        "attention_region": "center of the lesion",
        "max_attention": 0.96,
        "interpretation": "Model attention tightly focused with high gradient signal"
    }
}

print("testing with high confidence clean prediction...")
print("this should find no contradiction\n")

clean_output = run_critic(clean_state)
print("\ncritique:", clean_output['critique'])
print("contradictions:", clean_output['contradictions'])

clean_updated = {**clean_state, **clean_output}
clean_route = route_from_critic(clean_updated)
print("router went to:", clean_route)
print("expected: narrator (no contradiction)")

testing with high confidence clean prediction...
this should find no contradiction

  [Critic node] reviewing SHAP and Grad-CAM with LLM...
  [Critic node] done. contradictions=1, loop=0
    - The SHAP results show a relatively low mean SHAP value of 0.0045, which may not align with the high confidence score of 99.0% from the model, suggesting a potential mismatch between the model's confidence and the strength of evidence from SHAP.

critique: Agreement: Both SHAP and Grad-CAM indicate that the model focused on a specific localized region of the image, specifically the center of the lesion. Confidence assessment: The model's high confidence score of 99.0% appears to be supported by the tight focus of the model's attention, as indicated by the high max_attention value of 0.9600 from Grad-CAM. Next focus: Investigate the specific pixels highlighted by SHAP to determine if they align with the model's prediction and Grad-CAM's attention region, and assess if there are any other features i

In [20]:
import torch
from torchvision import transforms
from PIL import Image
import glob

import agents.graph as gm
importlib.reload(gm)
from agents.graph import xai_agent

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

images = glob.glob("../data/HAM10000_images_part_1/*.jpg")
test_tensor = transform(Image.open(images[3]).convert("RGB")).unsqueeze(0)

state = {
    "image_path":      images[3],
    "image_tensor":    test_tensor,
    "prediction":      "mel",
    "confidence":      0.55,
    "pred_class_idx":  1,
    "shap_result":     {},
    "gradcam_result":  {},
    "critique":        "",
    "contradictions":  [],
    "next_action":     "",
    "explanation":     "",
    "confidence_note": "",
    "loop_count":      0
}

print("running full graph with real LLM critic...")
result = xai_agent.invoke(state)

print("\ncritique from LLM:")
print(result['critique'])
print("\ncontradictions:", result['contradictions'])

XAI agent graph compiled successfully
running full graph with real LLM critic...
  [Planner] confidence=0.55, loop=0
  [Planner router] low confidence, going shap first
  [SHAP node] running SHAP analysis...
  [SHAP node] done. status=error, mean=0.000000
  [GradCAM node] running Grad-CAM analysis...
  [GradCAM node] done. status=success, region=top-left region of the image
  [Critic node] reviewing SHAP and Grad-CAM with LLM...
  [Critic node] done. contradictions=1, loop=0
    - SHAP analysis failed to run, preventing comparison with Grad-CAM and SHAP results
  [Critic router] contradiction found, looping back to planner
  [Planner] confidence=0.55, loop=1
  [Planner router] low confidence, going shap first
  [SHAP node] running SHAP analysis...
  [SHAP node] done. status=error, mean=0.000000
  [GradCAM node] running Grad-CAM analysis...
  [GradCAM node] done. status=success, region=top-left region of the image
  [Critic node] reviewing SHAP and Grad-CAM with LLM...
  [Critic node] d